In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import Row

In [0]:
df_brand = spark.table("databriks.silver.brands").drop("date_read","data_source","file_size")
df_category = spark.table("databriks.silver.category").drop("date_read","data_source","file_size")
df_product = spark.table("databriks.silver.products").drop("date_read","data_source","file_size")

In [0]:
joindf = df_product.join(df_category, on= "category_code", how="inner").join(df_brand, on= ["brand_code","category_code"], how="inner")
display(joindf)

In [0]:
joindf.write\
    .format("delta").option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable("databriks.gold.products")

In [0]:
df_customer = spark.table("databriks.silver.customers").drop("date_read","data_source","file_size")

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:

rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]        

In [0]:
df_region_mapping = spark.createDataFrame(rows)

In [0]:
df_customer = df_customer.join(df_region_mapping, on=['country', 'state'], how='left')

df_customer = df_customer.fillna({'region': 'Other'})

In [0]:
df_customer.write\
    .format("delta").option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable("databriks.gold.customers")

In [0]:
df_order = spark.table("databriks.silver.orders").drop("date_read","data_source","file_size")
display(df_order.limit(10))

In [0]:
df_order = df_order.withColumn("Total_price" , df_order["unit_price"] * df_order["quantity"])
df_order = df_order.withColumn("discount_amount" , F.round((df_order["unit_price"] * df_order["discount_pct"])/100,2))
df_order = df_order.withColumn("Total_discount" , df_order["discount_amount"] * df_order["quantity"])
df_order = df_order.withColumn("Total_price_after_discount" , df_order["Total_price"] - df_order["Total_discount"] + df_order["tax_amount"])
display(df_order)



In [0]:
df_order = df_order.withColumn("D_id", F.date_format(df_order["date"], "yyyyMMdd"))
display(df_order.limit(4))

In [0]:
df_order.write\
    .format("delta").option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable("databriks.gold.Orders")

In [0]:
df_date = spark.read.format("csv").option("header",True).option("inferSchema",True)\
    .load("s3://zg-databriks/date/*.csv")
display(df_date)

In [0]:
df_date = df_date.withColumn("D_id", F.date_format(df_date["date"], "yyyyMMdd")).dropDuplicates(["D_id"])

In [0]:
df_date = df_date.withColumn("day_name", F.initcap(df_date["day_name"]))
df_date = df_date.withColumn("week_of_year",F.abs(df_date["week_of_year"]))
df_date = df_date.withColumn("Q_year",F.concat(F.col("quarter"), F.lit("Q_"),F.col("year")))

display(df_date)


In [0]:
df_date.write\
    .format("delta").option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable("databriks.gold.date")

In [0]:
print('the end')